# RFC Simulator and Paper-Reproduction Notebook — Revised Presentation 29 Package

**Binder-safe patch of the original simulator.** This version keeps the dropdown UI when `ipywidgets` is available, removes the `pandas` dependency, and searches both the repository root and `notebooks/` for the JSON files.

It supports two clearly separated paths:

1. **Current revised Presentation 29 paper-reproduction path**: G, R, N, DownstreamPhysicalProjection, DimensionlessValidation, S, T, U, V, historical W, W2-R, W3, W4, CR-Trace, X, Y2, Z, QG.
2. **Legacy/development simulator path**: previous simulator modules A through Q, plus legacy G/N/R modules when present.

Expected files:

- `SimulationConfigs.json`
- `Module_G_R_N_S_T_FrozenPacket.json`
- `ValidationScreens_U_V_W_X_Y2_Z_QG.json` optional but recommended

Global rule for the paper-reproduction path: modules consume the frozen packet. They do not retune it. Historical W is retained as the scalar-lane Li7 diagnostic; W2-R/W3/W4/CR-Trace are the revised theorem additions.

**Mobile layout fix:** This build keeps the dropdown workflow, preserves the no-pandas/no-external-dependency design, and renders output as a vertical card stack instead of wide tables. Long rows now scroll downward on mobile instead of requiring side-to-side scrolling.

**Visualization update:** The notebook also adds automatically generated matplotlib charts for modules where visual comparisons improve interpretation. Charts are generated from the loaded JSON values; they do not retune or alter any RFC packet or validation result.


In [ ]:
import json
import math
from pathlib import Path

from IPython.display import display, Markdown, HTML

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception:
    widgets = None
    WIDGETS_AVAILABLE = False

BASE_DIR = Path.cwd()

EXPECTED_FILES = {
    "simulation_configs": "SimulationConfigs.json",
    "frozen_packet": "Module_G_R_N_S_T_FrozenPacket.json",
    "validation_screens": "ValidationScreens_U_V_W_X_Y2_Z_QG.json"
}

PAPER_REPRODUCTION_ORDER = [
    "G", "R", "N", "DownstreamPhysicalProjection", "DimensionlessValidation",
    "S", "T", "U", "V", "W", "W2R", "W3", "W4", "CRTrace", "X", "Y2", "Z", "QG"
]

LEGACY_DEVELOPMENT_ORDER = [
    "A", "B", "C", "D", "E", "F", "H", "I", "J", "K", "L", "M", "O", "P", "Q",
    "G_legacy", "N_legacy", "R_legacy"
]

LEGACY_MODULES = {"G_legacy", "N_legacy", "R_legacy"}
CANONICAL_ORDER = PAPER_REPRODUCTION_ORDER

# Mobile-first output mode. Set SHOW_CHARTS = False if you want text-only output.
MOBILE_VERTICAL_OUTPUT = True
SHOW_CHARTS = True
MAX_AUTO_CHARTS_PER_COMPONENT = 3

display(Markdown("## 1. Load repository JSON files"))
print("Working directory:", BASE_DIR)
print("Binder-safe patch: pandas removed; dropdown preserved when ipywidgets is available.")


In [ ]:
# RFC simulator display fix: mobile-first vertical cards, collapsible JSON, and optional charts.
RFC_DISPLAY_CSS = r'''
<style>
:root { --rfc-border:#c9c9c9; --rfc-bg:#fafafa; --rfc-head:#eeeeee; --rfc-soft:#f6f8fa; }
.rfc-section { max-width: 100%; }
.rfc-record-list { width:100%; max-width:100%; margin:0.6rem 0 1.2rem 0; }
.rfc-record { border:1px solid var(--rfc-border); border-radius:10px; background:var(--rfc-bg); margin:0.7rem 0; padding:10px 12px; max-width:100%; box-sizing:border-box; }
.rfc-record-title { font-weight:700; margin:0 0 0.45rem 0; padding-bottom:0.35rem; border-bottom:1px solid #e1e1e1; overflow-wrap:anywhere; }
.rfc-field { display:block; padding:0.38rem 0; border-top:1px solid #eeeeee; }
.rfc-field:first-of-type { border-top:0; }
.rfc-field-name { display:block; font-size:12px; font-weight:700; color:#555; margin-bottom:0.12rem; overflow-wrap:anywhere; }
.rfc-field-value { display:block; font-size:13px; line-height:1.42; overflow-wrap:anywhere; word-break:break-word; max-width:100%; }
.rfc-details summary { cursor:pointer; font-weight:600; color:#333; overflow-wrap:anywhere; }
.rfc-details pre, .rfc-pre { white-space:pre-wrap; overflow-wrap:anywhere; word-break:break-word; max-height:420px; overflow-y:auto; overflow-x:hidden; background:var(--rfc-soft); border:1px solid #ddd; border-radius:6px; padding:8px; margin:6px 0 0 0; font-size:12px; line-height:1.35; max-width:100%; }
.rfc-muted { color:#666; font-size:12px; }
.rfc-chip { display:inline-block; padding:2px 7px; border:1px solid #bbb; border-radius:999px; background:#f7f7f7; font-size:12px; margin:1px 3px 1px 0; }
.rfc-note { margin:0.45rem 0 0.9rem 0; }
.rfc-card { border:1px solid var(--rfc-border); border-radius:8px; padding:10px 12px; background:var(--rfc-bg); margin:0.6rem 0 1.2rem 0; max-width:100%; overflow-x:hidden; box-sizing:border-box; }
.rfc-error { border-left:4px solid #b00020; padding-left:10px; }
.rfc-ok { border-left:4px solid #238636; padding-left:10px; }
.rfc-warn { border-left:4px solid #bf8700; padding-left:10px; }
.rfc-chart-note { font-size:12px; color:#666; margin-top:-0.35rem; margin-bottom:0.4rem; }
/* Legacy table class retained for compatibility, but it now stacks and wraps on mobile. */
.rfc-table-wrap { width:100%; max-width:100%; overflow-x:hidden; overflow-y:visible; margin:0.6rem 0 1.2rem 0; }
table.rfc-table { border-collapse:collapse; width:100%; table-layout:auto; font-size:13px; line-height:1.35; }
table.rfc-table th, table.rfc-table td { border:1px solid var(--rfc-border); padding:8px; vertical-align:top; white-space:normal; overflow-wrap:anywhere; word-break:break-word; }
table.rfc-table th { background:var(--rfc-head); text-align:left; }
@media (max-width: 700px) {
  .rfc-record { padding:9px 10px; margin:0.65rem 0; }
  .rfc-field-name { font-size:11px; }
  .rfc-field-value { font-size:13px; }
  .rfc-details pre, .rfc-pre { max-height:360px; font-size:11px; }
}
</style>
'''
display(HTML(RFC_DISPLAY_CSS))


In [ ]:
def html_escape(value):
    text = str(value)
    return (text.replace("&", "&amp;")
                .replace("<", "&lt;")
                .replace(">", "&gt;")
                .replace('"', "&quot;"))


def shorten(value, max_len=260):
    """Return a compact plain-text preview. Nested values are converted to JSON first."""
    if isinstance(value, (dict, list)):
        value = json.dumps(value, ensure_ascii=False, indent=2)
    else:
        value = str(value)
    if len(value) > max_len:
        return value[:max_len] + " ..."
    return value


def value_type_label(value):
    if isinstance(value, dict):
        return f"dict[{len(value)}]"
    if isinstance(value, list):
        return f"list[{len(value)}]"
    if value is None:
        return "null"
    return type(value).__name__


def json_text(value):
    try:
        return json.dumps(value, ensure_ascii=False, indent=2)
    except Exception:
        return str(value)


def render_value_html(value, max_preview=220):
    """Render a value safely. Large/nested values become collapsible blocks."""
    if isinstance(value, (dict, list)):
        preview = html_escape(shorten(value, max_preview))
        full = html_escape(json_text(value))
        return (
            f"<details class='rfc-details'>"
            f"<summary>{html_escape(value_type_label(value))}: {preview}</summary>"
            f"<pre>{full}</pre>"
            f"</details>"
        )
    text = str(value)
    if len(text) > max_preview or "\n" in text:
        preview = html_escape(shorten(text, max_preview))
        full = html_escape(text)
        return (
            f"<details class='rfc-details'>"
            f"<summary>{preview}</summary>"
            f"<pre>{full}</pre>"
            f"</details>"
        )
    return html_escape(text)


def display_title(title):
    display(Markdown(f"## {title}"))


def display_note(text):
    display(Markdown(text))


def display_raw_json(obj, summary="Show raw JSON", max_height_px=420):
    full = html_escape(json_text(obj))
    display(HTML(
        f"<details class='rfc-details rfc-card'>"
        f"<summary>{html_escape(summary)}</summary>"
        f"<pre class='rfc-pre' style='max-height:{int(max_height_px)}px;'>{full}</pre>"
        f"</details>"
    ))


def infer_record_title(row, index):
    if not isinstance(row, dict):
        return f"Item {index + 1}"
    for key in ["module", "displayName", "field", "check", "constant", "symbol", "name", "item", "component"]:
        value = row.get(key)
        if value not in [None, ""]:
            return str(value)
    return f"Item {index + 1}"


def display_table(rows, max_rows=200):
    """Display rows as a mobile-first vertical card stack without pandas or sideways scrolling."""
    if rows is None:
        display_note("`None`")
        return
    if isinstance(rows, dict):
        rows = [{"field": k, "value": v} for k, v in rows.items()]
    if not isinstance(rows, list):
        display_note(f"`{html_escape(shorten(rows))}`")
        return
    if not rows:
        display_note("_No rows to display._")
        return

    normalized = []
    for item in rows[:max_rows]:
        normalized.append(item if isinstance(item, dict) else {"value": item})

    html = ["<div class='rfc-record-list'>"]
    for i, row in enumerate(normalized):
        title = html_escape(infer_record_title(row, i))
        html.append("<div class='rfc-record'>")
        html.append(f"<div class='rfc-record-title'>{title}</div>")
        for key, value in row.items():
            html.append("<div class='rfc-field'>")
            html.append(f"<span class='rfc-field-name'>{html_escape(key)}</span>")
            html.append(f"<span class='rfc-field-value'>{render_value_html(value)}</span>")
            html.append("</div>")
        html.append("</div>")
    html.append("</div>")
    if len(rows) > max_rows:
        html.append(f"<p class='rfc-muted'>Showing first {max_rows} rows of {len(rows)} total rows.</p>")
    display(HTML("".join(html)))


def display_dataframe_safe(rows, max_rows=200):
    # Compatibility name retained from the old notebook. No pandas required.
    display_table(rows, max_rows=max_rows)


In [ ]:
def candidate_file_paths(filename):
    cwd = Path.cwd()
    candidates = [
        cwd / filename,
        cwd / "notebooks" / filename,
        cwd.parent / filename,
        cwd.parent / "notebooks" / filename,
    ]
    try:
        candidates.extend(list(cwd.rglob(filename)))
    except Exception:
        pass
    seen = set()
    unique = []
    for path in candidates:
        key = str(path.resolve()) if path.exists() else str(path)
        if key not in seen:
            seen.add(key)
            unique.append(path)
    return unique


def locate_file(filename):
    for path in candidate_file_paths(filename):
        if path.exists() and path.is_file():
            return path
    return None


def load_json_file(filename, required=False):
    path = locate_file(filename)
    if path is None:
        if required:
            raise FileNotFoundError(f"Required file not found: {filename}")
        return None, {"file": filename, "path": "not found", "exists": False, "loaded": False, "error": None}
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data, {"file": filename, "path": str(path), "exists": True, "loaded": True, "error": None}
    except Exception as exc:
        if required:
            raise
        return None, {"file": filename, "path": str(path), "exists": True, "loaded": False, "error": str(exc)}


simulation_configs, simulation_status = load_json_file(EXPECTED_FILES["simulation_configs"])
frozen_packet, frozen_status = load_json_file(EXPECTED_FILES["frozen_packet"])
validation_file, validation_status = load_json_file(EXPECTED_FILES["validation_screens"])

display_table([simulation_status, frozen_status, validation_status])

if frozen_packet is None and simulation_configs is None:
    raise RuntimeError("No usable RFC JSON files were found. Put the JSON files in the repo root or notebooks/ folder.")

display(Markdown("Loaded files. Continuing with schema normalization."))


In [ ]:
def as_list(x):
    if x is None:
        return []
    return x if isinstance(x, list) else [x]


def normalize_name(name):
    return str(name).lower().replace("_", "").replace("-", "").replace(" ", "")


def unwrap_modules(config):
    if config is None:
        return []
    if isinstance(config, list):
        return config
    if isinstance(config, dict):
        for key in ["modules", "moduleConfigs", "simulationModules", "configs"]:
            if isinstance(config.get(key), list):
                return config[key]
    return []


simulation_modules = unwrap_modules(simulation_configs)


def module_name(module_obj):
    if not isinstance(module_obj, dict):
        return None
    for key in ["module", "id", "name", "moduleId", "moduleID"]:
        value = module_obj.get(key)
        if isinstance(value, str):
            return value
    return None


def available_config_module_names():
    return [name for obj in simulation_modules for name in [module_name(obj)] if name is not None]


def find_module_in_configs(name):
    target = normalize_name(name)
    for obj in simulation_modules:
        obj_name = module_name(obj)
        if obj_name and normalize_name(obj_name) == target:
            return obj
    return None


def find_key_contains(obj, tokens):
    if obj is None or not isinstance(obj, dict):
        return None
    tokens_low = [str(t).lower() for t in tokens]
    for key, value in obj.items():
        key_low = str(key).lower()
        if all(t in key_low for t in tokens_low):
            return value
    return None


def get_packet_module(name):
    if frozen_packet is None or not isinstance(frozen_packet, dict):
        return None
    target = normalize_name(name)
    direct_candidates = [name, str(name).lower(), f"module{name}", f"module_{name}", f"module{str(name).lower()}", f"module_{str(name).lower()}"]
    for key in direct_candidates:
        if key in frozen_packet:
            return frozen_packet[key]
    for key, value in frozen_packet.items():
        key_clean = normalize_name(key)
        if key_clean == target or key_clean == normalize_name("module" + str(name)):
            return value
        if key_clean.startswith(normalize_name("module" + str(name))):
            return value
    if target == "n":
        return find_key_contains(frozen_packet, ["module", "n"]) or find_key_contains(frozen_packet, ["dimensional"])
    if target == "r":
        return find_key_contains(frozen_packet, ["module", "r"]) or find_key_contains(frozen_packet, ["closure", "audit"])
    if target == "s":
        return find_key_contains(frozen_packet, ["module", "s"]) or find_key_contains(frozen_packet, ["anchor"])
    if target == "t":
        return find_key_contains(frozen_packet, ["module", "t"]) or find_key_contains(frozen_packet, ["coupling"])
    return None


def raw_validation_block():
    if isinstance(validation_file, dict):
        if isinstance(validation_file.get("validationScreens"), dict):
            return validation_file.get("validationScreens")
        return validation_file
    if isinstance(frozen_packet, dict) and isinstance(frozen_packet.get("validationScreens"), dict):
        return frozen_packet.get("validationScreens")
    return {}


def infer_screen_id(key):
    clean = normalize_name(key)
    if clean in ["u", "v", "w", "w2r", "w3", "w4", "crtrace", "x", "y2", "z", "qg"]:
        return clean.upper()
    for candidate in ["CRTrace", "Y2", "QG", "W2R", "W3", "W4", "U", "V", "W", "X", "Z"]:
        c = candidate.lower()
        if clean.startswith(c) or clean.startswith("module" + c) or ("module" + c) in clean:
            return candidate
    return str(key)


def normalize_validation_screens():
    raw = raw_validation_block()
    normalized = {}
    if isinstance(raw, dict):
        for key, value in raw.items():
            normalized[infer_screen_id(key)] = value
    for name in ["U", "V", "W", "W2R", "W3", "W4", "CRTrace", "X", "Y2", "Z", "QG"]:
        if name not in normalized:
            cfg = find_module_in_configs(name)
            if cfg is not None:
                normalized[name] = cfg
    return normalized


validation_screens = normalize_validation_screens()


def get_downstream_projection():
    if isinstance(frozen_packet, dict):
        for key in ["downstreamPhysicalProjectionScreen", "downstreamPhysicalProjection", "combinedDownstreamPhysicalProjectionScreen"]:
            if key in frozen_packet:
                return frozen_packet[key]
        found = find_key_contains(frozen_packet, ["downstream", "projection"])
        if found is not None:
            return found
    return find_module_in_configs("DownstreamPhysicalProjection")


def get_dimensionless_validation():
    if isinstance(frozen_packet, dict):
        for key in ["dimensionlessValidationLayer", "dimensionlessValidation", "dimensionlessObservableValidationLayer"]:
            if key in frozen_packet:
                return frozen_packet[key]
        found = find_key_contains(frozen_packet, ["dimensionless", "validation"])
        if found is not None:
            return found
    return find_module_in_configs("DimensionlessValidation")


def get_module_data(name):
    if name == "DownstreamPhysicalProjection":
        return get_downstream_projection()
    if name == "DimensionlessValidation":
        return get_dimensionless_validation()
    if name in ["U", "V", "W", "W2R", "W3", "W4", "CRTrace", "X", "Y2", "Z", "QG"]:
        return validation_screens.get(name) or find_module_in_configs(name)
    if name in ["G", "R", "N", "S", "T"]:
        return get_packet_module(name) or find_module_in_configs(name)
    return find_module_in_configs(name) or get_packet_module(name)


def build_all_runnable_modules():
    config_names = available_config_module_names()
    ordered = []
    for name in PAPER_REPRODUCTION_ORDER:
        if name not in ordered:
            ordered.append(name)
    for name in LEGACY_DEVELOPMENT_ORDER:
        if any(normalize_name(name) == normalize_name(cn) for cn in config_names) and name not in ordered:
            ordered.append(name)
    for name in config_names:
        if name not in ordered:
            ordered.append(name)
    return ordered


ALL_RUNNABLE_MODULES = build_all_runnable_modules()

display(Markdown("## 2. Normalized package status"))
summary_rows = []
for name in PAPER_REPRODUCTION_ORDER:
    summary_rows.append({"path": "paper_reproduction", "component": name, "found": get_module_data(name) is not None, "source": "frozen/validation/config auto-detected"})
for name in LEGACY_DEVELOPMENT_ORDER:
    if get_module_data(name) is not None:
        summary_rows.append({"path": "legacy_development", "component": name, "found": True, "source": "SimulationConfigs.json"})
display_table(summary_rows)
display(Markdown("Runnable module selector includes paper modules, legacy modules, and any extra modules found in `SimulationConfigs.json`."))


In [ ]:
def flatten_dict(obj, prefix=""):
    rows = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            next_prefix = f"{prefix}.{key}" if prefix else str(key)
            if isinstance(value, dict):
                # Keep nested objects collapsed rather than spreading huge JSON into overlapping cells.
                rows.append({"field": next_prefix, "type": value_type_label(value), "value": value})
            elif isinstance(value, list):
                rows.append({"field": next_prefix, "type": value_type_label(value), "value": value})
            else:
                rows.append({"field": next_prefix, "type": value_type_label(value), "value": value})
    else:
        rows.append({"field": prefix or "value", "type": value_type_label(obj), "value": obj})
    return rows


def flatten_dict_deep(obj, prefix="", max_depth=3, _depth=0):
    """Optional deeper flattening for diagnostics while still keeping large values collapsible."""
    rows = []
    if isinstance(obj, dict) and _depth < max_depth:
        for key, value in obj.items():
            next_prefix = f"{prefix}.{key}" if prefix else str(key)
            if isinstance(value, dict) and _depth + 1 < max_depth:
                rows.extend(flatten_dict_deep(value, next_prefix, max_depth=max_depth, _depth=_depth + 1))
            elif isinstance(value, list) and all(isinstance(x, dict) for x in value) and len(value) <= 12:
                rows.append({"field": next_prefix, "type": value_type_label(value), "value": value})
            else:
                rows.append({"field": next_prefix, "type": value_type_label(value), "value": value})
    else:
        rows.append({"field": prefix or "value", "type": value_type_label(obj), "value": obj})
    return rows


def find_first_record_list(obj):
    if isinstance(obj, list) and all(isinstance(x, dict) for x in obj):
        return obj
    if isinstance(obj, dict):
        for key in ["results", "screenResults", "selectedResults", "scoredConstants", "constants", "rows", "quarks", "mixingAngles", "observerBranchingResults", "neuralEEGTargets"]:
            if key in obj:
                found = find_first_record_list(obj[key])
                if found is not None:
                    return found
        for value in obj.values():
            found = find_first_record_list(value)
            if found is not None:
                return found
    return None


def extract_result_container(obj):
    if not isinstance(obj, dict):
        return obj
    for key in ["results", "screenResults", "selectedResults", "scoredConstants", "constants", "summary"]:
        if key in obj:
            return obj[key]
    return obj


def display_object(obj, title="Object"):
    display_title(title)
    if obj is None:
        display_note("**Not found.** Check that the relevant JSON file is present and populated.")
        return

    # Simple list of records: render directly as a table.
    if isinstance(obj, list):
        if all(isinstance(x, dict) for x in obj):
            display_table(obj)
        else:
            display_table([{"index": i, "value": v} for i, v in enumerate(obj)])
        display_raw_json(obj, "Show full raw list")
        return

    # Dictionaries: show a top-level readable summary first.
    if isinstance(obj, dict):
        summary_rows = []
        preferred_keys = [
            "module", "description", "currentStatus", "runMode", "canonicalRole",
            "consumesFrozenPacket", "mayRetuneFrozenPacket", "empiricalTargetsUsed",
            "parameterSearchPerformed", "mcmcUsed", "nutsUsed", "packetFrozen",
            "claimBoundary", "interpretation"
        ]
        used = set()
        for key in preferred_keys:
            if key in obj:
                summary_rows.append({"field": key, "type": value_type_label(obj[key]), "value": obj[key]})
                used.add(key)
        for key, value in obj.items():
            if key not in used:
                summary_rows.append({"field": key, "type": value_type_label(value), "value": value})
        display_table(summary_rows, max_rows=160)

        # If there is a clear record table inside, show it separately but do not duplicate giant raw output by default.
        record_list = find_first_record_list(obj)
        if record_list is not None and record_list is not obj and len(record_list) > 0:
            display_note(f"**Detected nested record table:** {len(record_list)} rows")
            display_table(record_list, max_rows=120)

        display_raw_json(obj, "Show full raw JSON for this component")
        return

    display(obj)


def numeric_or_none(value):
    try:
        if isinstance(value, bool):
            return None
        return float(value)
    except Exception:
        return None


def deep_get_by_key(obj, target_keys):
    target_low = {str(k).lower() for k in as_list(target_keys)}
    if isinstance(obj, dict):
        for key, value in obj.items():
            if str(key).lower() in target_low:
                return value
        for value in obj.values():
            found = deep_get_by_key(value, target_keys)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for value in obj:
            found = deep_get_by_key(value, target_keys)
            if found is not None:
                return found
    return None


def get_claim_boundary(obj):
    if isinstance(obj, dict):
        for key in ["claimBoundary", "boundary", "claimBoundarySummary", "interpretation", "status"]:
            if key in obj:
                return obj[key]
    return None


def print_boundary_if_present(obj):
    boundary = get_claim_boundary(obj)
    if boundary is not None:
        display_note("**Claim boundary / interpretation / status:**")
        display(HTML(f"<div class='rfc-card'>{render_value_html(boundary, max_preview=700)}</div>"))

In [ ]:
# Optional mobile-friendly visualizations generated from loaded JSON values.
def is_number_like(value):
    try:
        if isinstance(value, bool):
            return False
        float(value)
        return True
    except Exception:
        return False


def as_float(value):
    return float(value)


def nice_label(label, max_len=42):
    text = str(label).replace("_", " ").replace("over", "/").replace("Percent", " %")
    text = text.replace("Abs", "abs ").replace("V2", "V2 ").replace("Li7", "Li7")
    if len(text) > max_len:
        text = text[:max_len - 3] + "..."
    return text


def get_nested_dict(obj, *keys):
    if not isinstance(obj, dict):
        return None
    for key in keys:
        if isinstance(obj.get(key), dict):
            return obj[key]
    for key in keys:
        found = deep_get_by_key(obj, [key])
        if isinstance(found, dict):
            return found
    return None


def collect_numeric_items(d, keys=None, contains=None, exclude_contains=None):
    if not isinstance(d, dict):
        return []
    items = []
    if keys is not None:
        for key in keys:
            if key in d and is_number_like(d[key]):
                items.append((key, as_float(d[key])))
        return items
    contains = [str(x).lower() for x in (contains or [])]
    exclude_contains = [str(x).lower() for x in (exclude_contains or [])]
    for key, value in d.items():
        key_low = str(key).lower()
        if contains and not all(token in key_low for token in contains):
            continue
        if exclude_contains and any(token in key_low for token in exclude_contains):
            continue
        if is_number_like(value):
            items.append((key, as_float(value)))
    return items


def collect_error_items(results):
    if not isinstance(results, dict):
        return []
    items = []
    for key, value in results.items():
        if isinstance(value, dict):
            for err_key in ["absPercentError", "errorPercent", "absErrorPercent", "relativeDifferencePercent"]:
                if err_key in value and is_number_like(value[err_key]):
                    items.append((key, as_float(value[err_key])))
                    break
    return items


def chart_horizontal_bars(title, items, xlabel="Value", log=False, max_items=16):
    if not SHOW_CHARTS:
        return 0
    items = [(nice_label(k), v) for k, v in items if is_number_like(v)]
    if not items:
        return 0
    items = items[:max_items]
    labels = [k for k, _ in items]
    values = [float(v) for _, v in items]
    if log and any(v <= 0 for v in values):
        log = False
    try:
        import matplotlib.pyplot as plt
        height = max(2.8, 0.38 * len(labels) + 1.2)
        fig, ax = plt.subplots(figsize=(7.2, height))
        y = list(range(len(labels)))
        ax.barh(y, values)
        ax.set_yticks(y)
        ax.set_yticklabels(labels)
        ax.invert_yaxis()
        ax.set_xlabel(xlabel)
        ax.set_title(title)
        if log:
            ax.set_xscale("log")
            ax.set_xlabel(xlabel + " (log scale)")
        ax.grid(axis="x", alpha=0.25)
        fig.tight_layout()
        display(HTML("<div class='rfc-chart-note'>Chart generated directly from the loaded JSON values.</div>"))
        plt.show()
        return 1
    except Exception as exc:
        display_note(f"_Chart skipped: {exc}_")
        return 0


def chart_vertical_bars(title, items, ylabel="Value", log=False, max_items=10):
    if not SHOW_CHARTS:
        return 0
    items = [(nice_label(k, 24), v) for k, v in items if is_number_like(v)]
    if not items:
        return 0
    items = items[:max_items]
    labels = [k for k, _ in items]
    values = [float(v) for _, v in items]
    if log and any(v <= 0 for v in values):
        log = False
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(7.2, 3.9))
        x = list(range(len(labels)))
        ax.bar(x, values)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=35, ha="right")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        if log:
            ax.set_yscale("log")
            ax.set_ylabel(ylabel + " (log scale)")
        ax.grid(axis="y", alpha=0.25)
        fig.tight_layout()
        display(HTML("<div class='rfc-chart-note'>Chart generated directly from the loaded JSON values.</div>"))
        plt.show()
        return 1
    except Exception as exc:
        display_note(f"_Chart skipped: {exc}_")
        return 0


def chart_line(title, x_labels, series, ylabel="Value"):
    if not SHOW_CHARTS or not x_labels or not series:
        return 0
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(7.2, 3.9))
        x = list(range(len(x_labels)))
        for label, values in series:
            if values and all(is_number_like(v) for v in values):
                ax.plot(x, [float(v) for v in values], marker="o", label=label)
        ax.set_xticks(x)
        ax.set_xticklabels([nice_label(v, 18) for v in x_labels])
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.grid(alpha=0.25)
        ax.legend()
        fig.tight_layout()
        display(HTML("<div class='rfc-chart-note'>Chart generated directly from the loaded JSON values.</div>"))
        plt.show()
        return 1
    except Exception as exc:
        display_note(f"_Chart skipped: {exc}_")
        return 0


def display_visualizations_for_component(name, data):
    if not SHOW_CHARTS or data is None:
        return
    charts = 0
    display_note("### Visual summary")

    if name == "G":
        checks = get_nested_dict(data, "deterministicCheckValues")
        if checks:
            charts += chart_horizontal_bars("Module G deterministic check errors", collect_numeric_items(checks, contains=["error"]), "Absolute error")

    elif name == "R":
        src = get_nested_dict(data, "sourcePowerFractionsV2")
        triad = get_nested_dict(data, "interpretiveTriadFractions_WithDarkKernelBridge", "strictTriadFractions_NoBridge", "triadGroupPower")
        charts += chart_horizontal_bars("Module R source power fractions", collect_numeric_items(src), "Fraction")
        charts += chart_horizontal_bars("Module R triad grouping", collect_numeric_items(triad), "Fraction")

    elif name == "DownstreamPhysicalProjection":
        dark = get_nested_dict(data, "darkSectorProjection", "moduleA_DarkSector") or get_nested_dict(get_nested_dict(data, "results"), "moduleA_DarkSector")
        items = collect_numeric_items(dark, keys=["darkMatterDecayRatio", "darkEnergyTailRatio", "hubbleProxyRatio", "darkEnergyLateEarly", "darkMatterLateEarly"])
        charts += chart_vertical_bars("Downstream dark-sector ratios", items, "Late / early ratio")

    elif name == "DimensionlessValidation":
        results = get_nested_dict(data, "results") or data
        flags = get_nested_dict(results, "flags", "validationFlags") or get_nested_dict(data, "validationFlags")
        if isinstance(flags, dict):
            flag_items = [(k, 1 if v is True else 0) for k, v in flags.items() if isinstance(v, bool)]
            charts += chart_horizontal_bars("Dimensionless validation flags", flag_items, "Pass = 1")
        ratios = get_nested_dict(results, "dimensionlessRatios") or get_nested_dict(data, "dimensionlessRatios")
        charts += chart_horizontal_bars("Dimensionless ratios", collect_numeric_items(ratios), "Ratio", max_items=12)

    elif name == "T":
        raw = get_nested_dict(data, "rawInverseEnergyCoupling") or {}
        output = get_nested_dict(data, "output") or {}
        items = []
        if "alpha_N_raw_inverse" in raw:
            items.append(("raw inverse", raw["alpha_N_raw_inverse"]))
        for key in ["alpha_T_inverse", "reference_alpha_inverse"]:
            if key in output:
                items.append((key, output[key]))
        charts += chart_vertical_bars("Module T coupling map", items, "Inverse coupling")

    elif name in ["U", "V", "W"]:
        results = get_nested_dict(data, "results")
        errors = collect_error_items(results)
        log = name == "W" and any(v > 100 for _, v in errors)
        charts += chart_horizontal_bars(f"Module {name} percent-error screen", errors, "Absolute percent error", log=log)

    elif name == "W2R":
        results = get_nested_dict(data, "results") or {}
        li7_keys = ["oldScalarLi7OverObserved", "triadicLi7OverObserved", "strictNoBridgeLi7OverObserved", "interpretiveDarkBridgeLi7OverObserved", "QVOnlyLi7OverObserved", "CIFOnlyLi7OverObserved", "RFLOnlyLi7OverObserved", "NoQVRenormalizedLi7OverObserved"]
        charts += chart_horizontal_bars("W2-R Li7 over observed comparison", collect_numeric_items(results, keys=li7_keys), "Li7 / observed")
        drift_keys = ["DPercentDrift", "He3PercentDrift", "He4Drift", "maxLightElementAbsDriftPercent", "meanLightElementAbsDriftPercent"]
        charts += chart_horizontal_bars("W2-R light-element drift", collect_numeric_items(results, keys=drift_keys), "Percent drift")

    elif name == "W3":
        results = get_nested_dict(data, "results") or {}
        depths = ["depth6", "depth9", "depth12"]
        strict = [results.get("depth6StrictShapeCorrelation"), results.get("depth9StrictShapeCorrelation"), results.get("depth12StrictShapeCorrelation")]
        bridge = [results.get("depth6BridgeShapeCorrelation"), results.get("depth9BridgeShapeCorrelation"), results.get("depth12BridgeShapeCorrelation")]
        charts += chart_line("W3 solar grouped log-shape correlation by depth", depths, [("strict", strict), ("bridge", bridge)], "Correlation")
        mass_keys = ["V2Depth9H", "V2Depth9He", "V2Depth9CNO", "V2Depth9Alpha", "V2Depth9IronPeak", "V2Depth9sProcess", "V2Depth9rProcess", "V2Depth9RareTail"]
        charts += chart_horizontal_bars("W3 depth-9 grouped abundance state", collect_numeric_items(results, keys=mass_keys), "Mass fraction", log=True)

    elif name == "W4":
        results = get_nested_dict(data, "results") or {}
        family_keys = ["V2BridgeLiBeB_MAE", "V2BridgeCNO_MAE", "V2BridgeAlpha_MAE", "V2BridgeIronPeak_MAE", "V2BridgeSProcess_MAE", "V2BridgeRProcess_MAE"]
        charts += chart_horizontal_bars("W4 family residuals", collect_numeric_items(results, keys=family_keys), "Mean absolute residual (dex)")
        ablation_keys = ["NoSpallationLiBeB_MAE_Bridge", "NoHighZRaritySProcess_MAE_Bridge", "NoHighZRarityRProcess_MAE_Bridge"]
        charts += chart_horizontal_bars("W4 mechanism ablation penalties", collect_numeric_items(results, keys=ablation_keys), "MAE after ablation")

    elif name == "CRTrace":
        trace = get_nested_dict(data, "traceScores")
        charts += chart_horizontal_bars("CR-Trace score ledger", collect_numeric_items(trace), "Score")

    elif name == "X":
        results = get_nested_dict(data, "results") or {}
        eta = results.get("baryonEta_proxy") or results.get("baryonEtaProxy")
        if isinstance(eta, dict):
            items = []
            for k in ["rfc", "RFC", "reference"]:
                if k in eta and is_number_like(eta[k]):
                    items.append((k, eta[k]))
            charts += chart_vertical_bars("Module X baryon eta proxy", items, "Eta", log=True)

    elif name == "Y2":
        results = get_nested_dict(data, "results") or data
        q = get_nested_dict(results, "quarkMassScreen") or get_nested_dict(data, "quarkMassScreen")
        if isinstance(q, dict):
            quarks = q.get("quarks") if isinstance(q.get("quarks"), dict) else q
            q_errors = []
            for k, v in quarks.items() if isinstance(quarks, dict) else []:
                if isinstance(v, dict):
                    err = v.get("absPercentError") or v.get("errorPercent")
                    if is_number_like(err):
                        q_errors.append((k, err))
            charts += chart_horizontal_bars("Y2 quark mass percent errors", q_errors, "Absolute percent error")
        ckm = get_nested_dict(results, "CKMScreen") or get_nested_dict(data, "CKMScreen")
        pmns = get_nested_dict(results, "PMNSScreen") or get_nested_dict(data, "PMNSScreen")
        mix_items = []
        for label, block in [("CKM mean angle", ckm), ("PMNS mean angle", pmns)]:
            if isinstance(block, dict):
                for key in ["meanAngleErrorPercent", "CKMMeanAngleErrorPercent", "PMNSMeanAngleErrorPercent"]:
                    if key in block and is_number_like(block[key]):
                        mix_items.append((label, block[key]))
                        break
        charts += chart_horizontal_bars("Y2 mixing-screen mean angle errors", mix_items, "Absolute percent error")

    elif name == "Z":
        results = get_nested_dict(data, "results") or {}
        neural = get_nested_dict(results, "neuralEEGTargets")
        neural_errors = collect_error_items(neural)
        charts += chart_horizontal_bars("Module Z neural/EEG target percent errors", neural_errors, "Absolute percent error")

    elif name == "QG":
        results = get_nested_dict(data, "results") or {}
        qg_items = collect_numeric_items(results, keys=["finiteAmplitudeN18", "finiteAmplitudeN40", "relativeTail", "unitarityProxy", "refinementProxy", "geometryCoherence"])
        charts += chart_horizontal_bars("Module QG finite audit values", qg_items, "Value", log=True)

    if charts == 0:
        display_note("_No chart generated for this component; text/card output is the clearest representation._")


In [ ]:
def module_g_check():
    data = get_module_data("G")
    display_title("Module G: Deterministic Triadic Closure")
    if data is None:
        display_note("**Module G not found.**")
        return
    delta = numeric_or_none(deep_get_by_key(data, ["delta"]))
    cycle_length = numeric_or_none(deep_get_by_key(data, ["cycleLength", "cycle_length"]))
    phase_depth_k = numeric_or_none(deep_get_by_key(data, ["phaseDepthK", "phase_depth_k"]))
    alpha_packet = numeric_or_none(deep_get_by_key(data, ["alpha"]))
    nu_packet = numeric_or_none(deep_get_by_key(data, ["nu"]))
    epsilon_packet = numeric_or_none(deep_get_by_key(data, ["epsilon"]))
    empirical_targets = deep_get_by_key(data, ["empiricalTargetsUsed", "empirical_targets_used"])
    rows = []
    if delta is not None and cycle_length is not None and alpha_packet is not None:
        alpha_expected = math.log(delta) / cycle_length
        rows.append({"check": "alpha = log(delta) / cycleLength", "expected": alpha_expected, "packet": alpha_packet, "absError": abs(alpha_expected - alpha_packet)})
    if delta is not None and phase_depth_k is not None and nu_packet is not None:
        nu_expected = phase_depth_k * delta ** (-4)
        rows.append({"check": "nu = phaseDepthK * delta^(-4)", "expected": nu_expected, "packet": nu_packet, "absError": abs(nu_expected - nu_packet)})
    if alpha_packet is not None and nu_packet is not None and epsilon_packet is not None:
        epsilon_expected = alpha_packet * nu_packet
        rows.append({"check": "epsilon = alpha * nu", "expected": epsilon_expected, "packet": epsilon_packet, "absError": abs(epsilon_expected - epsilon_packet)})
    rows.append({"check": "empiricalTargetsUsed", "expected": False, "packet": empirical_targets, "absError": 0 if empirical_targets is False else "CHECK"})
    display_table(rows)
    display_object(data, "Module G Stored Packet")
    display_visualizations_for_component("G", data)


def module_r_check():
    data = get_module_data("R")
    display_title("Module R: Triad-Grouped Global Closure Audit")
    if data is None:
        display_note("**Module R not found.** This is a packaging problem if the paper reports Module R values.")
        return
    diagnostic_keys = ["rawRFLResidualScore", "sourceCoupledRFLResidualScore", "residualImprovement", "rawStandardizedResidualScore", "sourceCoupledRFLResidualScoreV2", "residualImprovementV2", "moduleRScoreV2", "cpResidual", "tailN18", "tailN40", "bestLagCorrelation", "bestLagCorrelationV1"]
    rows = []
    for key in diagnostic_keys:
        value = deep_get_by_key(data, [key])
        if value is not None:
            rows.append({"field": key, "value": value})
    if rows:
        display_table(rows)
    placeholder_flags = []
    for bad_key, bad_value in [("rawRFLResidualScore", 0.5), ("sourceCoupledRFLResidualScore", 0.5), ("residualImprovement", 0.0), ("tailN18", 0.0), ("tailN40", 0.0)]:
        val = numeric_or_none(deep_get_by_key(data, [bad_key]))
        if val is not None and abs(val - bad_value) < 1e-15:
            placeholder_flags.append(bad_key)
    if placeholder_flags:
        display_note("**WARNING:** possible placeholder Module R values detected: " + ", ".join(placeholder_flags))
    else:
        display_note("No obvious placeholder Module R values detected.")
    display_object(data, "Full Module R Object")
    display_visualizations_for_component("R", data)


def generic_module_check(name, title=None):
    data = get_module_data(name)
    display_object(data, title or f"Module {name}")
    display_visualizations_for_component(name, data)
    if name in LEGACY_MODULES or name in LEGACY_DEVELOPMENT_ORDER:
        display_note("**Legacy/development note:** This module is retained for continuity and exploration. It is not part of the current G/R/N/S/T paper-reproduction spine unless explicitly stated in the paper.")
    print_boundary_if_present(data)


def validation_screen_check(name):
    data = get_module_data(name)
    titles = {"U": "Module U: One-Anchor Constant Table Screen", "V": "Module V: Precision Cosmology Compressed-Parameter Screen", "W": "Module W: Historical Scalar-Lane BBN Diagnostic", "W2R": "Module W2-R: Revised Triadic-Weight BBN Closeout", "W3": "Module W3: Grouped Post-BBN Nucleosynthesis Cascade", "W4": "Module W4: Element-Resolved Solar-Abundance Screen", "CRTrace": "CR-Trace: Global Collapse-Rebirth Inheritance Trace Audit", "X": "Module X: CP/EDM Bound Screen", "Y2": "Module Y2: Particle-Sector Refinement Screen", "Z": "Module Z: Observer, Branching, Neural, and EEG Harness", "QG": "Module QG: Finite Spin-Foam Transition-Amplitude Audit"}
    display_object(data, titles.get(name, f"Module {name}"))
    display_visualizations_for_component(name, data)
    if name == "W":
        display_note("**Important:** W is the historical scalar-lane BBN diagnostic. It is retained to show the old Li7 wall and is superseded by W2-R.")
    if name == "Y2":
        display_note("**Important:** Y2 is exploratory candidate discovery. It is not final independent validation until frozen and retested as Y3.")
    if name == "CRTrace":
        display_note("**Important:** CR-Trace is an internal collapse-rebirth inheritance trace audit. It does not claim direct observation of a prior cosmic cycle.")
    print_boundary_if_present(data)


def run_component(name):
    if name == "G":
        module_g_check()
    elif name == "R":
        module_r_check()
    elif name == "N":
        generic_module_check("N", "Module N V2: Dimensional Projection Bridge")
    elif name == "S":
        generic_module_check("S", "Module S: One-Anchor SI Bridge")
    elif name == "T":
        generic_module_check("T", "Module T: Dimensionless Coupling Map")
    elif name == "DownstreamPhysicalProjection":
        generic_module_check("DownstreamPhysicalProjection", "Downstream Physical-Projection Screen")
    elif name == "DimensionlessValidation":
        generic_module_check("DimensionlessValidation", "Dimensionless Observable Validation Layer")
    elif name in ["U", "V", "W", "W2R", "W3", "W4", "CRTrace", "X", "Y2", "Z", "QG"]:
        validation_screen_check(name)
    else:
        generic_module_check(name, f"Module {name}")


def run_all_paper():
    display(Markdown("# RFC Full Paper-Reproduction Pass"))
    for name in PAPER_REPRODUCTION_ORDER:
        run_component(name)
        display(HTML("<hr>"))


def run_legacy_development_modules():
    display(Markdown("# RFC Legacy / Development Simulator Pass"))
    ran_any = False
    for name in LEGACY_DEVELOPMENT_ORDER:
        data = get_module_data(name)
        if data is not None:
            ran_any = True
            run_component(name)
            display(HTML("<hr>"))
    if not ran_any:
        display_note("No legacy/development modules were found in SimulationConfigs.json.")


def run_all():
    run_all_paper()


display(Markdown("## 3. Module runners loaded"))
display(Markdown("Use `run_all_paper()` for the current paper path, `run_legacy_development_modules()` for A-Q and legacy G/N/R, or the dropdown below."))


In [ ]:
def package_audit():
    display_title("Repository Package Audit")
    rows = []
    rows.append({"check": "SimulationConfigs.json loaded", "status": simulation_configs is not None})
    rows.append({"check": "Frozen packet JSON loaded", "status": frozen_packet is not None})
    rows.append({"check": "Standalone validation screens JSON loaded", "status": validation_file is not None})
    for name in ["G", "R", "N", "S", "T", "DownstreamPhysicalProjection", "DimensionlessValidation", "U", "V", "W", "W2R", "W3", "W4", "CRTrace", "X", "Y2", "Z", "QG"]:
        rows.append({"check": f"{name} found", "status": get_module_data(name) is not None})
    legacy_found = [name for name in LEGACY_DEVELOPMENT_ORDER if get_module_data(name) is not None]
    rows.append({"check": "Legacy/development modules found", "status": len(legacy_found) > 0})
    display_table(rows)
    missing = [row["check"] for row in rows if row.get("status") is False]
    if missing:
        display_note("**Packaging warnings:**")
        for item in missing:
            display_note("- " + item)
    else:
        display_note("**Package audit passed:** all expected paper-reproduction components were found.")
    if validation_file is None:
        display_note("**Recommendation:** add `ValidationScreens_U_V_W_X_Y2_Z_QG.json` as a standalone file, even if the same screens are mirrored in the frozen packet.")
    if legacy_found:
        display_note("**Legacy/development modules detected:** " + ", ".join(legacy_found))
    else:
        display_note("No legacy/development modules detected. This is okay only if SimulationConfigs.json intentionally omits them.")


package_audit()


In [ ]:
if WIDGETS_AVAILABLE:
    display(Markdown("## 4. Interactive module selector"))
    selector = widgets.Dropdown(
        options=ALL_RUNNABLE_MODULES,
        value="G" if "G" in ALL_RUNNABLE_MODULES else ALL_RUNNABLE_MODULES[0],
        description="Component:",
        layout=widgets.Layout(width="100%")
    )
    run_button = widgets.Button(description="Run selected component", button_style="primary")
    run_paper_button = widgets.Button(description="Run paper reproduction", button_style="success")
    run_legacy_button = widgets.Button(description="Run legacy/development", button_style="warning")
    output = widgets.Output()

    def on_run_clicked(_):
        with output:
            output.clear_output()
            run_component(selector.value)

    def on_run_paper_clicked(_):
        with output:
            output.clear_output()
            run_all_paper()

    def on_run_legacy_clicked(_):
        with output:
            output.clear_output()
            run_legacy_development_modules()

    run_button.on_click(on_run_clicked)
    run_paper_button.on_click(on_run_paper_clicked)
    run_legacy_button.on_click(on_run_legacy_clicked)

    display(widgets.VBox([selector, widgets.VBox([run_button, run_paper_button, run_legacy_button])]))
    display(output)
else:
    display(Markdown("## 4. Interactive widgets unavailable"))
    display(Markdown("The dropdown requires `ipywidgets`. Manual commands still work: `run_component('G')`, `run_all_paper()`, or `run_legacy_development_modules()`."))


## Manual commands

```python
package_audit()
run_all_paper()
run_component('G')
run_component('R')
run_component('N')
run_component('DownstreamPhysicalProjection')
run_component('DimensionlessValidation')
run_component('S')
run_component('T')
run_component('U')
run_component('V')
run_component('W')
run_component('W2R')
run_component('W3')
run_component('W4')
run_component('CRTrace')
run_component('X')
run_component('Y2')
run_component('Z')
run_component('QG')
run_legacy_development_modules()
run_component('A')
run_component('B')
run_component('Q')
run_component('G_legacy')
run_component('N_legacy')
run_component('R_legacy')
```

Interpretation reminders:

- Module G is the current frozen deterministic packet source.
- Module R audits the frozen packet; it does not create the packet.
- Module U is a first-pass electromagnetic/atomic constant table screen.
- Module W is a historical scalar-lane BBN diagnostic superseded by W2-R.
- W2-R is the revised triadic-weight BBN closeout.
- W3/W4 are internal nucleosynthesis cascade and element-resolved screens.
- CR-Trace is an internal collapse-rebirth inheritance trace audit, not direct observation of a prior cosmic cycle.
- Module Y2 is exploratory candidate discovery and must be frozen/retested as Y3 before independent validation claims.
- A-Q and legacy G/N/R modules are retained for development continuity and should not be confused with the current paper-reproduction spine.

- Mobile output is rendered as vertical cards to avoid side-to-side scrolling.
- Charts are generated from the loaded JSON values and do not change the frozen packet or validation results.
